# PicMap v2 — Colab Launcher

**Workflow:** Edit scripts locally in VS Code → push to GitHub → open this notebook → Run All

This notebook always pulls the latest code from GitHub before running.

In [ ]:
# ── Cell 1: Mount Google Drive ──────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── Cell 2: Clone / pull latest code from GitHub ────────────────────────
import os

REPO_URL = 'https://github.com/cmprice1/picmap.git'
BRANCH   = 'v2-colab'
REPO_DIR = '/content/picmap'

if os.path.exists(REPO_DIR):
    print('Repo exists — pulling latest...')
    !git -C {REPO_DIR} pull origin {BRANCH}
else:
    print('Cloning repo...')
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}

!echo "Latest commit: $(git -C {REPO_DIR} log -1 --format='%h %s')"

In [ ]:
# ── Cell 3: Install dependencies ────────────────────────────────────────
!pip install -q -r {REPO_DIR}/requirements.txt
print('Dependencies ready.')

In [ ]:
# ── Cell 4: Run build.py ────────────────────────────────────────────────
# The default --takeout path is already set in build.py.
# Override here if needed.

TAKEOUT = '/content/drive/My Drive/PicMap-V2 Project/Takeout'
OUTPUT  = f'{REPO_DIR}/output'
CONFIG  = f'{REPO_DIR}/config.json'

!python {REPO_DIR}/build.py \
    --takeout "{TAKEOUT}" \
    --output  "{OUTPUT}" \
    --config  "{CONFIG}"

In [ ]:
# ── Cell 5: Push results to GitHub ──────────────────────────────────────
# Configure git identity (only needed once per Colab session)
!git -C {REPO_DIR} config user.email "colab@picmap.dev"
!git -C {REPO_DIR} config user.name "Colab Runner"

# Stage and commit output
!git -C {REPO_DIR} add output/data.json output/photos/
!git -C {REPO_DIR} commit -m "Generated output from Colab" || echo "Nothing to commit."
!git -C {REPO_DIR} push origin {BRANCH}

In [ ]:
# ── Cell 6: Preview results (optional) ─────────────────────────────────
import json

with open(f'{REPO_DIR}/output/data.json') as f:
    data = json.load(f)

print(f"Trip: {data['trip']['title']}")
print(f"Stops: {len(data['stops'])}")
print(f"Waypoints: {len(data['waypoints'])}")
print(f"Route coords: {len(data['route']['geometry']['coordinates'])}")
print()
for s in data['stops']:
    print(f"  {s['order']}. {s['name']} ({s['type']}, {len(s['photos'])} photos)")